<a href="https://colab.research.google.com/github/vedavikas07-V/FUNDAMENTALS-OF-DATASCIENCE/blob/main/CO5%20AT2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# ============================================================
# LOGISTIC REGRESSION WITH MANUAL CLASS-WEIGHTED LOSS
# Compare Unweighted vs Weighted Logistic Regression
# ============================================================

import numpy as np
import pandas as pd
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, recall_score, accuracy_score

# ------------------------------------------------------------
# 1. Create a severely imbalanced dataset (98:2)
# ------------------------------------------------------------

X, y = make_classification(
    n_samples=5000,
    n_features=5,
    n_informative=4,
    n_redundant=0,
    n_classes=2,
    weights=[0.98, 0.02],
    flip_y=0,
    random_state=42
)

print("Original class distribution:")
print(pd.Series(y).value_counts())
print()

# ------------------------------------------------------------
# 2. Train-test split
# ------------------------------------------------------------

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.30,
    stratify=y,
    random_state=42
)

# ------------------------------------------------------------
# 3. Standardize features manually
# ------------------------------------------------------------

mean = X_train.mean(axis=0)
std = X_train.std(axis=0)

X_train = (X_train - mean) / (std + 1e-8)
X_test = (X_test - mean) / (std + 1e-8)

# Add intercept/bias column
X_train = np.c_[np.ones(X_train.shape[0]), X_train]
X_test = np.c_[np.ones(X_test.shape[0]), X_test]


# ------------------------------------------------------------
# 4. Sigmoid function
# ------------------------------------------------------------

def sigmoid(z):
    z = np.clip(z, -500, 500)
    return 1 / (1 + np.exp(-z))


# ------------------------------------------------------------
# 5. Logistic Regression from scratch
# ------------------------------------------------------------

def train_logistic_regression(
    X,
    y,
    learning_rate=0.05,
    epochs=3000,
    class_weights=None
):
    n_samples, n_features = X.shape

    # Initialize parameters
    weights = np.zeros(n_features)

    for epoch in range(epochs):

        # Prediction
        z = np.dot(X, weights)
        probabilities = sigmoid(z)

        # Error
        error = probabilities - y

        # ----------------------------------------------------
        # Apply manually calculated class weights
        # ----------------------------------------------------
        if class_weights is not None:
            sample_weights = np.where(
                y == 0,
                class_weights[0],
                class_weights[1]
            )

            error = error * sample_weights

        # Gradient
        gradient = np.dot(X.T, error) / n_samples

        # Update parameters
        weights -= learning_rate * gradient

    return weights


# ------------------------------------------------------------
# 6. Calculate class weights MANUALLY
# ------------------------------------------------------------

class_counts = np.bincount(y_train)

n_samples = len(y_train)
n_classes = len(class_counts)

weight_0 = n_samples / (n_classes * class_counts[0])
weight_1 = n_samples / (n_classes * class_counts[1])

class_weights = {
    0: weight_0,
    1: weight_1
}

print("Class counts in training data:")
print("Class 0:", class_counts[0])
print("Class 1:", class_counts[1])
print()

print("Manually calculated class weights:")
print("Weight for class 0:", round(weight_0, 4))
print("Weight for class 1:", round(weight_1, 4))
print()


# ------------------------------------------------------------
# 7. Train UNWEIGHTED Logistic Regression
# ------------------------------------------------------------

weights_unweighted = train_logistic_regression(
    X_train,
    y_train,
    learning_rate=0.05,
    epochs=3000,
    class_weights=None
)


# ------------------------------------------------------------
# 8. Train WEIGHTED Logistic Regression
# ------------------------------------------------------------

weights_weighted = train_logistic_regression(
    X_train,
    y_train,
    learning_rate=0.05,
    epochs=3000,
    class_weights=class_weights
)


# ------------------------------------------------------------
# 9. Prediction function
# ------------------------------------------------------------

def predict(X, weights, threshold=0.5):
    probabilities = sigmoid(np.dot(X, weights))
    predictions = (probabilities >= threshold).astype(int)
    return predictions


# ------------------------------------------------------------
# 10. Predictions
# ------------------------------------------------------------

y_pred_unweighted = predict(
    X_test,
    weights_unweighted
)

y_pred_weighted = predict(
    X_test,
    weights_weighted
)


# ------------------------------------------------------------
# 11. Confusion matrices
# ------------------------------------------------------------

cm_unweighted = confusion_matrix(
    y_test,
    y_pred_unweighted
)

cm_weighted = confusion_matrix(
    y_test,
    y_pred_weighted
)


# ------------------------------------------------------------
# 12. Minority-class recall
# ------------------------------------------------------------

recall_unweighted = recall_score(
    y_test,
    y_pred_unweighted,
    pos_label=1
)

recall_weighted = recall_score(
    y_test,
    y_pred_weighted,
    pos_label=1
)


# ------------------------------------------------------------
# 13. Accuracy
# ------------------------------------------------------------

accuracy_unweighted = accuracy_score(
    y_test,
    y_pred_unweighted
)

accuracy_weighted = accuracy_score(
    y_test,
    y_pred_weighted
)


# ------------------------------------------------------------
# 14. Display results
# ------------------------------------------------------------

print("=" * 60)
print("UNWEIGHTED LOGISTIC REGRESSION")
print("=" * 60)

print("\nConfusion Matrix:")
print(cm_unweighted)

print("\nAccuracy:",
      round(accuracy_unweighted, 4))

print("Minority Class Recall:",
      round(recall_unweighted, 4))


print("\n" + "=" * 60)
print("CLASS-WEIGHTED LOGISTIC REGRESSION")
print("=" * 60)

print("\nConfusion Matrix:")
print(cm_weighted)

print("\nAccuracy:",
      round(accuracy_weighted, 4))

print("Minority Class Recall:",
      round(recall_weighted, 4))


# ------------------------------------------------------------
# 15. Comparison
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("COMPARISON")
print("=" * 60)

print(
    f"Unweighted Minority Recall : {recall_unweighted:.4f}"
)

print(
    f"Weighted Minority Recall   : {recall_weighted:.4f}"
)

improvement = recall_weighted - recall_unweighted

print(
    f"Recall Improvement         : {improvement:.4f}"
)

print(
    f"Unweighted Accuracy        : {accuracy_unweighted:.4f}"
)

print(
    f"Weighted Accuracy          : {accuracy_weighted:.4f}"
)

Original class distribution:
0    4900
1     100
Name: count, dtype: int64

Class counts in training data:
Class 0: 3430
Class 1: 70

Manually calculated class weights:
Weight for class 0: 0.5102
Weight for class 1: 25.0

UNWEIGHTED LOGISTIC REGRESSION

Confusion Matrix:
[[1470    0]
 [  30    0]]

Accuracy: 0.98
Minority Class Recall: 0.0

CLASS-WEIGHTED LOGISTIC REGRESSION

Confusion Matrix:
[[1012  458]
 [   7   23]]

Accuracy: 0.69
Minority Class Recall: 0.7667

COMPARISON
Unweighted Minority Recall : 0.0000
Weighted Minority Recall   : 0.7667
Recall Improvement         : 0.7667
Unweighted Accuracy        : 0.9800
Weighted Accuracy          : 0.6900
